In [1]:
%reset -f

In [2]:
from pynq import PL
from pynq import (allocate, Overlay)
import numpy as np
from PIL import Image

PL.reset()

In [3]:
ol = Overlay('resizer-zcu102.bit')

In [4]:
# help(ol)

In [5]:
img2axis = ol.img2axis_0

In [6]:
# help(img2axis.register_map)

In [7]:
def img_to_axis(ip,buffer, eos,frame_cnt):
    
# Configure registers:
    # Write physical address of buffer to data_port register
    ip.register_map.data_port = buffer.physical_address

    # Set end_of_stream 
    ip.register_map.end_of_stream = eos

    # Set frame_no to 88
    ip.register_map.frame_cnt = frame_cnt
    # Start the IP core by setting the ap_start bit in CTRL register
    ip.register_map.CTRL.AP_START=1

In [8]:


def image_to_RGB(image_fname):
    # === LOAD AND CONVERT IMAGE TO RGB ===
    img = Image.open(f"{image_fname}").convert('RGB') 
    img_np = np.array(img)  # Shape: (H, W, 3), dtype=uint8

    # === PACK RGB TO INT32 ===
    # Format: 0x00RRGGBB (most significant byte can be 0)
    r = img_np[:, :, 0].astype(np.uint32)
    g = img_np[:, :, 1].astype(np.uint32)
    b = img_np[:, :, 2].astype(np.uint32)
    rgb_packed = (b << 16) | (g << 8) | r  # Shape: (H, W)

    # Allocate contiguous buffer with dtype uint32
    buffer = allocate(shape=rgb_packed.shape, dtype=np.uint32)

    # Copy packed pixels into buffer
    np.copyto(buffer, rgb_packed)

    print(f"Packed buffer shape: {buffer.shape}, dtype: {buffer.dtype}")
    return buffer

In [9]:
from pynq.lib.video import *
vdma = ol.axi_vdma_0

vdma.readchannel.reset()
vdma.readchannel.mode = VideoMode(width=1920, height=1080, bits_per_pixel=24)

vdma.readchannel.start()



In [10]:
print(f"VDMA.running={vdma.readchannel.running},\nVDMA.activeframe={vdma.readchannel.activeframe},\nVDMA.mode={vdma.readchannel.mode}")

VDMA.running=True,
VDMA.activeframe=0,
VDMA.mode=VideoMode: width=1920 height=1080 bpp=24 fps=60


In [11]:
img_fname='1920x1080-full-hd-nature-landscape.jpg'

In [12]:
buff_o=image_to_RGB(img_fname)

Packed buffer shape: (1080, 1920), dtype: uint32


In [13]:
img_to_axis(ol.img2axis_0,buff_o,True,4)

In [ ]:
frame = vdma.readchannel.readframe()

In [15]:
print(f"type(frame)={type(frame)},frame.shape={frame.shape},frame.dtype={frame.dtype}")

type(frame)=<class 'pynq.buffer.PynqBuffer'>,frame.shape=(1080, 1920, 3),frame.dtype=uint8


In [16]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

#Convert to NumPy array
frame_np = np.array(frame)
#save image
img = Image.fromarray(frame_np, 'RGB')
#img.save(f"{timestamp}-ouput.png")
img.save(f"image-output.png")

# playground

In [17]:
hasattr(ol.axi_vdma_0, 'write')  # should return True


True

In [18]:
img_to_axis(ol.img2axis_0,buff_o,True,88)

In [19]:
help(VideoMode)

Help on class VideoMode in module pynq.lib.video.common:

class VideoMode(builtins.object)
 |  VideoMode(width, height, bits_per_pixel, fps=60, stride=None)
 |  
 |  Class for holding the information about a video mode
 |  
 |  Attributes
 |  ----------
 |  height : int
 |      Height of the video frame in lines
 |  width : int
 |      Width of the video frame in pixels
 |  stride : int
 |      Width of a line in the video frame in bytes
 |  bits_per_pixel : int
 |      Bits per pixel
 |  bytes_per_Pixel : int
 |      Bytes required to represent each pixel
 |  shape : tuple of int
 |      Numpy-style tuple describing the video frame
 |  
 |  Methods defined here:
 |  
 |  __eq__(self, mode)
 |      Return self==value.
 |  
 |  __init__(self, width, height, bits_per_pixel, fps=60, stride=None)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __repr__(self)
 |      Return repr(self).
 |  
 |  -----------------------------------------------------------------

In [20]:
dir(ol.axi_vdma_0)

['MM2SChannel',
 'S2MMChannel',
 '_FrameList',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_call',
 '_fullpath',
 '_gpio',
 '_interrupts',
 '_register_name',
 '_registers',
 '_setup_packet_prototype',
 '_start_ert',
 '_start_sw',
 'bindto',
 'device',
 'framecount',
 'mmio',
 'read',
 'readchannel',
 'register_map',
 's2mm_introut',
 'signature',
 'write']

In [21]:
ol.axi_vdma_0.framecount

2

In [22]:
len(ol.axi_vdma_0.readchannel._frames)

2